# AS1 PPO Sequential Experiments, Evaluation, and Report Evidence

This notebook trains and evaluates four Proximal Policy Optimization (PPO) configurations in the AS1 Godot 2D Platform Collector. The experiment queue runs one model at a time because every Godot environment uses local WebSocket port `8765`.

### Recommended run order

1. Run **Setup** and **Environment validation**.
2. Run the **random baseline** once.
3. Review the four configurations in the experiment plan.
4. Run the training queue. When one model finishes and is saved, the next starts automatically.
5. Evaluate every saved model using the same 50-episode protocol.
6. Generate the comparison tables and report evidence graphs.

Completed models are skipped by default, so an interrupted notebook can be restarted without retraining configurations that already finished.

## 1. Experiment definition

### Observation space (10 normalized values)

| Index | Observation |
|---:|---|
| 0–1 | Player x/y position |
| 2–3 | Horizontal/vertical velocity |
| 4 | Whether the player is on the floor |
| 5–6 | Relative x/y position of the nearest apple |
| 7–8 | Relative x/y position of the nearest snail |
| 9 | Fraction of apples remaining |

### Discrete action space

| Value | Action |
|---:|---|
| 0 | Do nothing |
| 1 | Move left |
| 2 | Move right |
| 3 | Jump |
| 4 | Move left and jump |
| 5 | Move right and jump |

### Reward design

| Event | Reward |
|---|---:|
| Each decision step | −0.01 |
| Collect an apple | +10 |
| Collect all apples | +100 |
| Hit an enemy | −20 |
| Fall | −20 |
| Reach 1,000-step time limit | −5 |
| Move toward/away from nearest apple | clipped distance shaping, −0.1 to +0.1 |

## 2. Setup and reproducibility

In [1]:
import platform
import random
import sys
import time
from pathlib import Path

import gymnasium
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import stable_baselines3
import torch
import websockets
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_random_seed(SEED)

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Gymnasium:", gymnasium.__version__)
print("Stable-Baselines3:", stable_baselines3.__version__)
print("PyTorch:", torch.__version__)
print("WebSockets:", websockets.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Apple MPS available:", torch.backends.mps.is_available())

Python executable: /Users/booxuan/miniconda3/envs/rl-ppo/bin/python
Python version: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
Platform: macOS-26.6.1-arm64-arm-64bit
Gymnasium: 1.3.0
Stable-Baselines3: 2.9.0
PyTorch: 2.13.0
WebSockets: 17.0.1
NumPy: 2.4.6
Pandas: 3.0.5
Matplotlib: 3.11.1
Apple MPS available: True


### 2.1 Project paths and queue controls

In [ ]:
# Locate the repository from the notebook directory or any parent directory.
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (
        location
        for location in search_locations
        if (location / "GodotGameEngineScene" / "as_1" / "project.godot").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root from the current directory")

AS1_DIRECTORY = PROJECT_ROOT / "GodotGameEngineScene" / "as_1"
AGENT_DIRECTORY = AS1_DIRECTORY / "agent"
CONFIG_DIRECTORY = AGENT_DIRECTORY / "ppo_configs"
RESULTS_DIRECTORY = AS1_DIRECTORY / "results"
MODELS_DIRECTORY = AS1_DIRECTORY / "models"
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)

if str(AGENT_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(AGENT_DIRECTORY))

from as1_env import AS1GodotEnv

# Queue controls. Review these values before starting a long run.
HEADLESS = True
TRAIN_EXPERIMENTS = True
SKIP_COMPLETED_MODELS = True
SMOKE_TEST_TIMESTEPS = 5_000
TARGET_TOTAL_TIMESTEPS = 200_000
RANDOM_BASELINE_EPISODES = 50
EVALUATION_EPISODES = 50

print("Project root:", PROJECT_ROOT)
print("Configuration directory:", CONFIG_DIRECTORY)
print("Results directory:", RESULTS_DIRECTORY)
print("Models directory:", MODELS_DIRECTORY)
print("AS1GodotEnv imported successfully")

Project root: /Users/booxuan/Documents/UCCD2063-Artificial-Intelligence-Techniques-RL-Godot
Configuration directory: /Users/booxuan/Documents/UCCD2063-Artificial-Intelligence-Techniques-RL-Godot/GodotGameEngineScene/as_1/agent/ppo_configs
Results directory: /Users/booxuan/Documents/UCCD2063-Artificial-Intelligence-Techniques-RL-Godot/GodotGameEngineScene/as_1/results
Models directory: /Users/booxuan/Documents/UCCD2063-Artificial-Intelligence-Techniques-RL-Godot/GodotGameEngineScene/as_1/models
AS1GodotEnv imported successfully


## 3. Reusable experiment functions

Keeping episode collection and metric calculation in functions ensures that the random and PPO policies are evaluated in the same way.

In [3]:
def run_episodes(env, episodes, *, model=None, seed=SEED):
    # Evaluate a random policy or a supplied model for complete episodes.
    records = []
    observation, info = env.reset(seed=seed)

    for episode in range(1, episodes + 1):
        episode_reward = 0.0
        episode_steps = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            if model is None:
                action = env.action_space.sample()
            else:
                action, _state = model.predict(observation, deterministic=True)

            observation, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            episode_steps += 1

        records.append(
            {
                "episode": episode,
                "reward": episode_reward,
                "steps": episode_steps,
                "reason": info.get("reason", "unknown"),
                "apples_collected": info.get("apples_collected", 0),
                "completed": info.get("reason") == "success",
            }
        )

        if episode < episodes:
            observation, info = env.reset()

    return pd.DataFrame(records)


def summarize_results(results_df):
    # Return report-ready aggregate metrics for one policy.
    return {
        "episodes": len(results_df),
        "mean_reward": results_df["reward"].mean(),
        "reward_std": results_df["reward"].std(),
        "mean_steps": results_df["steps"].mean(),
        "mean_apples": results_df["apples_collected"].mean(),
        "completion_rate": results_df["completed"].mean(),
    }


def save_figure(filename):
    # Save the active figure at report quality and then display it.
    plt.tight_layout()
    plt.savefig(RESULTS_DIRECTORY / filename, dpi=300, bbox_inches="tight")
    plt.show()

## 4. Validate the Gymnasium–Godot interface

In [4]:
validation_env = AS1GodotEnv(headless=HEADLESS)
try:
    check_env(validation_env, warn=True)
    print("Gymnasium environment validation passed")
finally:
    validation_env.close()

Godot Engine v4.6.3.stable.official.7d41c59c4 - https://godotengine.org
Metal 4.0 - Forward+ - Using Device #0: Apple - Apple M3 (Apple9)

Gymnasium environment validation passed


## 5. Random-agent baseline

This is the control condition. A trained agent should improve mean reward, apples collected, and completion rate relative to this policy.

In [ ]:
random_env = AS1GodotEnv(headless=HEADLESS)
random_env.action_space.seed(SEED)
try:
    random_results_df = run_episodes(
        random_env,
        RANDOM_BASELINE_EPISODES,
        model=None,
    )
finally:
    random_env.close()

random_results_df.to_csv(RESULTS_DIRECTORY / "random_agent_results.csv", index=False)
random_summary = summarize_results(random_results_df)
display(random_results_df)
pd.Series(random_summary)

Godot Engine v4.6.3.stable.official.7d41c59c4 - https://godotengine.org
Metal 4.0 - Forward+ - Using Device #0: Apple - Apple M3 (Apple9)



## 6. PPO experiment queue

The baseline is followed by three controlled configurations. Each experiment changes only one PPO hyperparameter, while environment design, seed, training budget, and evaluation procedure remain fixed.

### Hyperparameter experiment design

Three PPO hyperparameters are investigated using two possible values each:

| Hyperparameter | Baseline value | Alternative value |
|---|---:|---:|
| Learning rate | 0.0003 | 0.0001 |
| Entropy coefficient (`ent_coef`) | 0.01 | 0.02 |
| PPO clip range (`clip_range`) | 0.20 | 0.10 |

A complete factorial search would require \(2^3 = 8\) configurations. However, this experiment uses a one-factor-at-a-time design to reduce training time and make the effect of each hyperparameter easier to interpret.

| Configuration | Learning rate | Entropy coefficient | Clip range | Included? |
|---|---:|---:|---:|:---:|
| Baseline | 0.0003 | 0.01 | 0.20 | Yes |
| Lower learning rate | 0.0001 | 0.01 | 0.20 | Yes |
| Higher exploration | 0.0003 | 0.02 | 0.20 | Yes |
| Smaller PPO clip | 0.0003 | 0.01 | 0.10 | Yes |
| Low learning rate + high entropy | 0.0001 | 0.02 | 0.20 | No |
| Low learning rate + small clip | 0.0001 | 0.01 | 0.10 | No |
| High entropy + small clip | 0.0003 | 0.02 | 0.10 | No |
| All alternative values | 0.0001 | 0.02 | 0.10 | No |

The four included configurations change only one hyperparameter relative to the baseline. This controlled design allows performance differences to be linked more clearly to the parameter being investigated. Combined parameter interactions are outside the scope of the initial experiment because a full search would approximately double the required training time.

In [ ]:
BASE_PPO_CONFIG = {
    "learning_rate": 0.0003,
    "n_steps": 1024,
    "batch_size": 64,
    "n_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.20,
    "ent_coef": 0.01,
}

# Read every numbered JSON file in ppo_configs. Add another file to extend the queue.
import json

config_files = sorted(CONFIG_DIRECTORY.glob("*.json"))
if not config_files:
    raise FileNotFoundError(f"No PPO configuration files found in {CONFIG_DIRECTORY}")

EXPERIMENTS = []
for config_file in config_files:
    experiment = json.loads(config_file.read_text(encoding="utf-8"))
    required_fields = {"name", "label", "change", "overrides"}
    missing_fields = required_fields.difference(experiment)
    if missing_fields:
        raise ValueError(f"{config_file.name} is missing fields: {sorted(missing_fields)}")
    unknown_parameters = set(experiment["overrides"]).difference(BASE_PPO_CONFIG)
    if unknown_parameters:
        raise ValueError(
            f"{config_file.name} contains unknown PPO parameters: {sorted(unknown_parameters)}"
        )
    experiment["config_file"] = config_file.name
    EXPERIMENTS.append(experiment)

experiment_names = [experiment["name"] for experiment in EXPERIMENTS]
if len(experiment_names) != len(set(experiment_names)):
    raise ValueError("Every PPO configuration must have a unique name")

experiment_rows = []
for position, experiment in enumerate(EXPERIMENTS, start=1):
    resolved_config = {**BASE_PPO_CONFIG, **experiment["overrides"]}
    experiment_rows.append(
        {
            "queue_position": position,
            "name": experiment["name"],
            "label": experiment["label"],
            "controlled_change": experiment["change"],
            "config_file": experiment["config_file"],
            **resolved_config,
        }
    )

experiment_plan_df = pd.DataFrame(experiment_rows)
experiment_plan_df.to_csv(RESULTS_DIRECTORY / "experiment_plan.csv", index=False)
experiment_plan_df

## 7. Sequential training queue

The queue saves every configuration under a unique name. After a model completes, its environment is closed and the next experiment starts automatically. With `SKIP_COMPLETED_MODELS = True`, existing `.zip` models are preserved and skipped.

The smoke stage and full stage belong to the **same model**. PPO may finish slightly above the requested timestep target because it collects complete rollouts of 1,024 transitions.

In [ ]:
import gc

training_metadata = []

for queue_position, experiment in enumerate(EXPERIMENTS, start=1):
    name = experiment["name"]
    label = experiment["label"]
    model_path = MODELS_DIRECTORY / f"as1_ppo_{name}"
    model_zip_path = model_path.with_suffix(".zip")
    monitor_path = RESULTS_DIRECTORY / f"{name}_monitor.csv"
    metadata_path = RESULTS_DIRECTORY / f"{name}_training_metadata.csv"
    resolved_config = {**BASE_PPO_CONFIG, **experiment["overrides"]}

    print(f"\n[{queue_position}/{len(EXPERIMENTS)}] {label}")

    if SKIP_COMPLETED_MODELS and model_zip_path.exists():
        print("Completed model already exists; skipping training:", model_zip_path)
        if metadata_path.exists():
            training_metadata.append(pd.read_csv(metadata_path).iloc[0].to_dict())
        continue

    if not TRAIN_EXPERIMENTS:
        print("TRAIN_EXPERIMENTS is False; training not started")
        continue

    training_env = Monitor(
        AS1GodotEnv(headless=HEADLESS),
        filename=str(monitor_path),
        info_keywords=("reason", "apples_collected", "total_reward"),
    )
    training_start = time.perf_counter()

    try:
        model = PPO(
            policy="MlpPolicy",
            env=training_env,
            tensorboard_log=str(RESULTS_DIRECTORY / "tensorboard" / name),
            device="auto",
            seed=SEED,
            verbose=1,
            **resolved_config,
        )

        # Stage A: short end-to-end smoke test.
        model.learn(
            total_timesteps=SMOKE_TEST_TIMESTEPS,
            progress_bar=True,
            tb_log_name=name,
        )
        model.save(MODELS_DIRECTORY / f"as1_ppo_{name}_smoke")

        # Stage B: continue the same model to the common training budget.
        remaining_timesteps = max(TARGET_TOTAL_TIMESTEPS - model.num_timesteps, 0)
        if remaining_timesteps:
            model.learn(
                total_timesteps=remaining_timesteps,
                reset_num_timesteps=False,
                progress_bar=True,
                tb_log_name=name,
            )

        model.save(model_path)
        record = {
            "queue_position": queue_position,
            "name": name,
            "label": label,
            "seed": SEED,
            "timesteps": model.num_timesteps,
            "training_seconds": time.perf_counter() - training_start,
            **resolved_config,
        }
        pd.DataFrame([record]).to_csv(metadata_path, index=False)
        training_metadata.append(record)
        print("Saved completed model:", model_zip_path)
    finally:
        training_env.close()
        if "model" in locals():
            del model
        gc.collect()

if training_metadata:
    training_metadata_df = pd.DataFrame(training_metadata)
    training_metadata_df.to_csv(
        RESULTS_DIRECTORY / "all_training_metadata.csv",
        index=False,
    )
    display(training_metadata_df)

print("\nTraining queue finished.")

## 8. Evaluate every saved configuration

Every model is evaluated over 50 deterministic episodes using the same environment and metrics. Evaluation begins only after the training queue cell has finished.

In [ ]:
evaluation_frames = []
evaluation_summaries = []

for queue_position, experiment in enumerate(EXPERIMENTS, start=1):
    name = experiment["name"]
    label = experiment["label"]
    model_path = MODELS_DIRECTORY / f"as1_ppo_{name}"
    model_zip_path = model_path.with_suffix(".zip")

    if not model_zip_path.exists():
        print(f"Skipping evaluation; model not found: {model_zip_path}")
        continue

    print(f"Evaluating [{queue_position}/{len(EXPERIMENTS)}] {label}")
    model = PPO.load(model_path)
    evaluation_env = AS1GodotEnv(headless=HEADLESS)

    try:
        result_df = run_episodes(
            evaluation_env,
            EVALUATION_EPISODES,
            model=model,
        )
    finally:
        evaluation_env.close()

    result_df.insert(0, "name", name)
    result_df.insert(1, "label", label)
    result_df.to_csv(RESULTS_DIRECTORY / f"{name}_evaluation.csv", index=False)
    evaluation_frames.append(result_df)

    summary = summarize_results(result_df)
    summary.update(
        {
            "queue_position": queue_position,
            "name": name,
            "label": label,
            "seed": SEED,
            **BASE_PPO_CONFIG, **experiment["overrides"],
        }
    )
    evaluation_summaries.append(summary)
    pd.DataFrame([summary]).to_csv(
        RESULTS_DIRECTORY / f"{name}_summary.csv",
        index=False,
    )

    del model
    gc.collect()

if not evaluation_frames:
    raise RuntimeError("No completed model was available for evaluation")

all_evaluation_df = pd.concat(evaluation_frames, ignore_index=True)
experiment_summary_df = pd.DataFrame(evaluation_summaries).sort_values("queue_position")
all_evaluation_df.to_csv(RESULTS_DIRECTORY / "all_evaluations.csv", index=False)
experiment_summary_df.to_csv(RESULTS_DIRECTORY / "all_experiment_summaries.csv", index=False)

display(experiment_summary_df[
    ["label", "mean_reward", "reward_std", "mean_steps", "mean_apples", "completion_rate"]
].round(3))

### Previous single-baseline run snapshot

Before the four-configuration queue was added, the notebook recorded the following baseline results at 200,704 timesteps. These values are historical context only; the new experiment CSV files are the evidence for the current run.

| Policy | Mean reward | Reward SD | Mean steps | Mean apples | Completion rate |
|---|---:|---:|---:|---:|---:|
| Random (10 episodes) | −21.8328 | — | 335.5 | 0.0 | 0% |
| PPO baseline (50 deterministic episodes) | −4.1237 | 0.0000 | 1000.0 | 1.0 | 0% |

The old PPO policy improved reward and collected one apple but did not solve the level. All deterministic episodes ended by timeout.

## 9. Report-ready comparison table

The table combines the random control with every completed PPO configuration. Report mean and standard deviation together, then discuss task-specific measures such as apples and completion rate.

In [ ]:
random_comparison_row = {
    "label": "Random agent",
    **random_summary,
}
comparison_df = pd.concat(
    [
        pd.DataFrame([random_comparison_row]),
        experiment_summary_df,
    ],
    ignore_index=True,
    sort=False,
)
comparison_df.to_csv(RESULTS_DIRECTORY / "policy_comparison.csv", index=False)
comparison_df[
    ["label", "episodes", "mean_reward", "reward_std", "mean_steps", "mean_apples", "completion_rate"]
].round(3)

## 10. Report evidence graphs

### 10.1 Training learning curves — essential

The graph compares learning progress across configurations. Raw episode rewards are faint; rolling means reveal the overall trend. A missing line means its monitor CSV has not been generated yet.

In [ ]:
plt.figure(figsize=(11, 6))
plotted_training_runs = 0

for experiment in EXPERIMENTS:
    monitor_path = RESULTS_DIRECTORY / f"{experiment['name']}_monitor.csv"
    if not monitor_path.exists():
        print("Training log not found:", monitor_path.name)
        continue

    monitor_df = pd.read_csv(monitor_path, comment="#")
    if monitor_df.empty:
        continue

    monitor_df["cumulative_timesteps"] = monitor_df["l"].cumsum()
    rolling_window = min(50, max(5, len(monitor_df) // 10))
    monitor_df["rolling_reward"] = monitor_df["r"].rolling(
        rolling_window,
        min_periods=1,
    ).mean()

    plt.plot(
        monitor_df["cumulative_timesteps"],
        monitor_df["rolling_reward"],
        linewidth=2,
        label=experiment["label"],
    )
    plotted_training_runs += 1

if plotted_training_runs == 0:
    plt.close()
    raise FileNotFoundError("No monitor CSV files are available; run the training queue first")

plt.title("PPO Training Reward by Configuration")
plt.xlabel("Cumulative training timesteps")
plt.ylabel("Rolling mean episode reward")
plt.grid(alpha=0.25)
plt.legend()
save_figure("training_reward_comparison.png")

### 10.2 Random agent versus all PPO configurations — essential

The panels use separate scales so reward, apples, and completion rate remain readable.

In [ ]:
plot_comparison_df = comparison_df.set_index("label")
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [
    ("mean_reward", "Mean reward"),
    ("mean_apples", "Mean apples collected"),
    ("completion_rate", "Completion rate"),
]
colors = ["gray", "steelblue", "darkorange", "seagreen", "mediumpurple"]

for axis, (column, label) in zip(axes, metrics):
    values = plot_comparison_df[column]
    bars = axis.bar(range(len(values)), values, color=colors[: len(values)])
    axis.set_title(label)
    axis.set_xticks(range(len(values)), plot_comparison_df.index, rotation=25, ha="right")
    axis.grid(axis="y", alpha=0.25)
    axis.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    if column == "completion_rate":
        axis.set_ylim(0, max(1.0, values.max() * 1.15))

fig.suptitle("Random Baseline versus PPO Configurations")
save_figure("all_policy_metric_comparison.png")

### 10.3 Terminal outcomes across configurations — essential for diagnosing failure

In [ ]:
random_outcomes = random_results_df[["reason"]].copy()
random_outcomes["label"] = "Random agent"
ppo_outcomes = all_evaluation_df[["reason", "label"]].copy()
all_outcomes = pd.concat([random_outcomes, ppo_outcomes], ignore_index=True)

outcome_order = ["success", "enemy", "fall", "timeout"]
outcome_comparison = pd.crosstab(
    all_outcomes["reason"],
    all_outcomes["label"],
    normalize="columns",
).reindex(outcome_order).fillna(0.0)

outcome_comparison.plot(kind="bar", figsize=(12, 6))
plt.title("Terminal Outcome Rates by Policy Configuration")
plt.xlabel("Terminal reason")
plt.ylabel("Proportion of episodes")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.legend(title="Policy", bbox_to_anchor=(1.02, 1), loc="upper left")
save_figure("all_terminal_outcomes.png")

### 10.4 Evaluation consistency by configuration — supporting graph

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

for experiment in EXPERIMENTS:
    subset = all_evaluation_df[all_evaluation_df["name"] == experiment["name"]]
    if subset.empty:
        continue
    axes[0].plot(
        subset["episode"],
        subset["reward"],
        marker="o",
        markersize=2,
        linewidth=1,
        label=experiment["label"],
    )
    axes[1].plot(
        subset["episode"],
        subset["apples_collected"],
        marker="o",
        markersize=2,
        linewidth=1,
        label=experiment["label"],
    )

axes[0].set_ylabel("Total reward")
axes[0].set_title("Deterministic PPO Evaluation by Episode")
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].set_xlabel("Evaluation episode")
axes[1].set_ylabel("Apples collected")
axes[1].grid(alpha=0.25)
axes[1].legend()
save_figure("all_evaluation_consistency.png")

## 11. Automatic result checks

These checks identify whether each configuration solved the task and whether deterministic evaluation lacks variation.

In [ ]:
for _, result in experiment_summary_df.iterrows():
    print(f"\n{result['label']}:")
    if result["completion_rate"] == 0:
        print("• No successful completion; do not claim the task was solved.")
    else:
        print(f"• Completion rate: {result['completion_rate']:.1%}")
    if result["reward_std"] == 0:
        print("• Identical deterministic rewards; multiple training seeds are recommended.")
    if result["mean_reward"] > random_summary["mean_reward"]:
        print("• Mean reward improved relative to the random policy.")
    if result["mean_apples"] > random_summary["mean_apples"]:
        print("• More apples were collected than by the random policy.")

best_result = experiment_summary_df.loc[experiment_summary_df["mean_reward"].idxmax()]
print(
    f"\nBest mean reward: {best_result['label']} "
    f"({best_result['mean_reward']:.3f})"
)

## 12. Experiment interpretation and validation

The four configurations were initially trained using seed `42`. Their performance should be compared using the following measures, in priority order:

1. Completion rate
2. Mean apples collected
3. Mean evaluation reward
4. Reward standard deviation
5. Mean episode length and terminal outcomes

The best configuration should not be selected using reward alone. A configuration that completes the level or collects more apples may be preferable even when its mean reward is slightly lower.

After identifying the best initial configuration, it should be retrained as a new model using additional seeds such as `123` and `456`. The final performance should then be reported as the mean ± standard deviation across the three independent seed runs.

### Results interpretation

Complete this subsection after all four configurations have been trained and evaluated:

- **Best configuration:** To be determined
- **Completion rate:** To be determined
- **Mean apples collected:** To be determined
- **Mean evaluation reward:** To be determined
- **Most frequent terminal outcome:** To be determined
- **Effect of the changed hyperparameter:** To be discussed
- **Recommended configuration for multiple-seed validation:** To be determined

One 200,000-step model previously required approximately 3.6 hours. Therefore, the initial four-configuration experiment may require approximately 14–16 hours, excluding evaluation. Completed models are saved separately and skipped when the training queue is restarted.

## 13. Assignment report material checklist

### Methodology evidence

- Environment screenshot and short level/task description
- Observation table, action table, reward table, and termination conditions
- PPO architecture/algorithm explanation
- Four-configuration experiment-plan and hyperparameter table
- Training budget, hardware/software versions, seed, and evaluation protocol
- Simple Python–WebSocket–Godot interaction diagram

### Results evidence

- Training learning-curve comparison across all configurations
- Random-versus-PPO metric comparison
- Terminal-outcome comparison
- Evaluation consistency graph
- Summary table with mean, standard deviation, apples, steps, and completion rate
- Training-time table and saved raw episode CSV files

### Discussion evidence

- Identify the best configuration using more than reward alone
- Explain whether completion rate and apples improved
- Discuss flat deterministic results or 0% completion honestly
- Relate differences to the one hyperparameter changed in each experiment
- State the one-seed limitation and propose multiple-seed validation

Training-console screenshots are optional appendix material. They do not replace CSV results, learning curves, comparison tables, or interpretation.